In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_from_mysql
from pyspark.sql import functions as F

cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

fact_transactions = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_transactions")
bridge            = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "bridge_client_account")
dim_client        = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "dim_client")
dim_date          = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "dim_date")
fact_loans        = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_loans")

## Client transaction profile (attributed via OWNER disposition)

In [ ]:
from src.client_analytics import build_client_transaction_profile

client_profile = build_client_transaction_profile(fact_transactions, bridge, dim_date)

print("Clients with at least one owned-account transaction:", client_profile.count())
print("Total clients in dim_client:", dim_client.count())

client_profile.orderBy(F.desc("total_transactions")).show(10)

In [ ]:
from pydoc import cli
from src.client_analytics import add_behavioral_features

client_profile.cache()

snapshot_date = client_profile.agg(F.max("last_transaction_date")).collect()[0][0]
print("Snapshot_date:", snapshot_date)

client_profile = add_behavioral_features(client_profile, snapshot_date)

client_profile.select(
    "client_id", "total_transactions", "active_days",
    "avg_transactions_per_month", "recency_days"
).orderBy(F.desc("avg_transactions_per_month")).show(10)

In [ ]:
from src.client_analytics import add_client_initiated_recency

client_profile = add_client_initiated_recency(
    client_profile, fact_transactions, bridge, dim_date, snapshot_date
)

client_profile.select("recency_days_real").summary("min", "25%", "50%", "75%", "max").show()

In [ ]:
from pyspark.sql import Window

recency_window = Window.orderBy(F.col("recency_days_real").desc())

frequency_window = Window.orderBy(F.col("avg_transactions_per_month").asc())

monetary_window = Window.orderBy(F.col("net_flow").asc())

client_rfm = (client_profile
    .withColumn("r_score", F.ntile(4).over(recency_window))
    .withColumn("f_score", F.ntile(4).over(frequency_window))
    .withColumn("m_score", F.ntile(4).over(monetary_window))
    .withColumn("rfm_score", F.col("r_score") + F.col("f_score") + F.col("m_score"))
)

In [ ]:
client_rfm.orderBy(
    F.desc("rfm_score"), F.desc("net_flow")
).select(
    "client_id", "r_score", "f_score", "m_score", "rfm_score", "net_flow"
).show(10)

In [ ]:
from src.client_analytics import build_loan_status_flags

loan_flags = build_loan_status_flags(fact_loans, bridge)

client_rfm_loans = (client_rfm
    .join(loan_flags, "client_id", "left")
    .withColumn("has_problem_loan", F.coalesce(F.col("has_problem_loan"), F.lit(0)))
    .withColumn("loan_count", F.coalesce(F.col("loan_count"), F.lit(0)))
)

In [ ]:
client_rfm_loans = client_rfm_loans.withColumn(
    "rfm_segment",
    F.when(F.col("rfm_score") >= 10, "Champions")
     .when(F.col("rfm_score") >= 7, "Loyal")
     .when(F.col("rfm_score") >= 5, "At Risk")
     .otherwise("Lost")
)

In [ ]:
(client_rfm_loans
    .groupBy("rfm_segment")
    .pivot("has_problem_loan", [0, 1])
    .count()
    .show())

In [ ]:
(client_rfm_loans
    .groupBy("rfm_segment")
    .agg(
        F.count("*").alias("n_clients"),
        F.round(F.avg("has_problem_loan") * 100, 1).alias("pct_problem_loan")
    )
    .orderBy(F.desc("pct_problem_loan"))
    .show())